# 02 · Our model vs LendingClub's grade

**The question:** can a model that only sees the application rank loans by risk better than LendingClub's own grades?

**The fair-test rules**

1. **Train on the past, test on the future.** The model learns from loans issued 2007–2014 and is tested on 2015 loans it has never seen.
2. **No peeking at the answer.** The model never sees LendingClub's grade or interest rate (the thing we compete against), or anything that happened after the loan was made.
3. **Two baselines.** LendingClub's sub-grade and its interest rate, used as risk rankings.

**How we score it:** AUC, the chance that the model rates a randomly picked defaulted loan riskier than a randomly picked repaid one. 0.5 is a coin flip, 1.0 is perfect.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_parquet("data/loans_clean.parquet")
print(f"{len(df):,} loans loaded")

In [ ]:
NUMERIC = [
    "loan_amnt", "fico", "annual_inc", "dti", "revol_bal", "revol_util", "open_acc",
    "total_acc", "delinq_2yrs", "inq_last_6mths", "pub_rec", "emp_years", "credit_history_years",
]
CATEGORICAL = ["home_ownership", "purpose"]
FEATURES = NUMERIC + CATEGORICAL
# Deliberately NOT features: grade, sub_grade, int_rate (LendingClub's answer),
# loan_status, total_pymnt, funded_amnt (what happened later), job_title (saved for the stereotype test)

train = df[df["year"] <= 2014]
test = df[df["year"] == 2015].copy()

print(f"Train: {len(train):,} loans (2007-2014), default rate {train['default'].mean():.1%}")
print(f"Test:  {len(test):,} loans (2015),      default rate {test['default'].mean():.1%}")

## Train two models

- **Logistic regression:** simple and easy to explain.
- **Gradient boosting:** a stronger model that learns from many small decision trees.

Both get the same preparation: fill missing numbers with the middle value, and turn categories like `purpose` into yes/no columns. Training the boosting model takes a minute or two.

In [ ]:
prep = ColumnTransformer([
    ("numbers", Pipeline([("fill", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC),
    ("categories", Pipeline([
        ("fill", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), CATEGORICAL),
])

models = {
    "Logistic regression": Pipeline([("prep", clone(prep)), ("model", LogisticRegression(max_iter=2000))]),
    "Gradient boosting": Pipeline([("prep", clone(prep)), ("model", HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.05, random_state=42))]),
}

risk = {}
for name, pipe in models.items():
    pipe.fit(train[FEATURES], train["default"])
    risk[name] = pipe.predict_proba(test[FEATURES])[:, 1]
    print(f"Trained: {name}")

## The head-to-head on 2015 loans

In [ ]:
SUBGRADE_ORDER = [f"{g}{n}" for g in "ABCDEFG" for n in range(1, 6)]
subgrade_rank = test["sub_grade"].map({sg: i for i, sg in enumerate(SUBGRADE_ORDER)})

scores = {name: roc_auc_score(test["default"], r) for name, r in risk.items()}
scores["LendingClub sub-grade (baseline)"] = roc_auc_score(test["default"], subgrade_rank)
scores["LendingClub interest rate (baseline)"] = roc_auc_score(test["default"], test["int_rate"])

results = pd.Series(scores, name="AUC on 2015 loans").sort_values(ascending=False).round(3).to_frame()
results

**Leak check before celebrating.** On data like this, a model that beats the platform's grade by a huge margin usually means an after-the-fact column slipped in. If your model is far above the grade, re-check the feature list before trusting it.

## Does the model know something the grade doesn't?

Two loans with the same sub-grade pay the same interest rate. If our model can still tell which of them is riskier, it adds information the grade misses. We measure AUC **inside each sub-grade** and average it, weighted by how many loans each has. 0.5 here would mean the model adds nothing beyond the grade.

In [ ]:
# We decided BEFORE looking at 2015 results that gradient boosting is our main model.
# Picking whichever model scored best on 2015 would be peeking at the test.
BEST = "Gradient boosting"
test["model_risk"] = risk[BEST]
print("Main model:", BEST)

within = []
for sg, group in test.groupby("sub_grade"):
    if group["default"].nunique() == 2 and len(group) >= 200:
        within.append((sg, len(group), roc_auc_score(group["default"], group["model_risk"])))
within = pd.DataFrame(within, columns=["sub_grade", "loans", "auc"])
weighted_auc = np.average(within["auc"], weights=within["loans"])
print(f"Average AUC inside a sub-grade: {weighted_auc:.3f}  (0.5 = adds nothing beyond the grade)")

## The money test

Imagine investing in 2015. Option 1: pick a grade and buy all its loans. Option 2: pick the same grade, but only buy the half our model rates safest. Same grades, same interest rates, so any difference comes from our model.

**Simple return** = (total paid back − amount lent) ÷ amount lent. It ignores timing and fees, so treat it as approximate.

In [ ]:
test["simple_return"] = (test["total_pymnt"] - test["funded_amnt"]) / test["funded_amnt"]
# Inside each sub-grade, the "safest half" is every loan at or below that sub-grade's median model risk
test["safest_half"] = test.groupby("sub_grade")["model_risk"].transform(lambda s: s <= s.median())

def summarise(frame):
    out = frame.groupby("grade").agg(
        loans=("default", "size"),
        default_rate_pct=("default", "mean"),
        simple_return_pct=("simple_return", "mean"),
    )
    out[["default_rate_pct", "simple_return_pct"]] *= 100
    return out

all_loans = summarise(test)
safest = summarise(test[test["safest_half"]])
riskiest = summarise(test[~test["safest_half"]])

money = pd.DataFrame({
    "Default %: all loans": all_loans["default_rate_pct"],
    "Default %: safest half": safest["default_rate_pct"],
    "Default %: riskiest half": riskiest["default_rate_pct"],
    "Return %: all loans": all_loans["simple_return_pct"],
    "Return %: safest half": safest["simple_return_pct"],
})
money.loc["All grades"] = [
    test["default"].mean() * 100,
    test.loc[test["safest_half"], "default"].mean() * 100,
    test.loc[~test["safest_half"], "default"].mean() * 100,
    test["simple_return"].mean() * 100,
    test.loc[test["safest_half"], "simple_return"].mean() * 100,
]
money = money.round(1)
money

In [ ]:
grades = list(all_loans.index)
x = np.arange(len(grades))
width = 0.36
BLUE, ORANGE = "#2a78d6", "#eb6834"   # validated colour pair
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

overall_all = money.loc["All grades", "Default %: all loans"]
overall_safe = money.loc["All grades", "Default %: safest half"]

fig, ax = plt.subplots(figsize=(9, 4.8), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")
ax.bar(x - width / 2 - 0.01, all_loans["default_rate_pct"], width, color=BLUE, label="All loans in the grade")
ax.bar(x + width / 2 + 0.01, safest["default_rate_pct"], width, color=ORANGE, label="Safest half, picked by our model")

ax.set_xticks(x, grades)
ax.set_xlabel("LendingClub grade (A = safest)", color=MUTED)
ax.set_ylabel("Loans that defaulted (%)", color=MUTED)
ax.set_title(
    f"Same grades, same interest rates: the safest half defaulted {overall_safe:.1f}% vs {overall_all:.1f}%",
    loc="left", color=INK, fontsize=12, pad=26,
)
ax.text(0, 1.025, "2015 loans the model never saw", transform=ax.transAxes, color=MUTED, fontsize=9)
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")
ax.tick_params(colors=MUTED)
ax.legend(frameon=False, loc="upper left")
plt.tight_layout()

Path("outputs").mkdir(exist_ok=True)
fig.savefig("outputs/money_test.png", dpi=200)
plt.show()

## Where the model fails

Judges score honesty. These are 2015 loans our model rated among the very safest that still defaulted. Look at them and write one sentence in the README about why the model missed them.

In [ ]:
surprises = test[test["default"] == 1].nsmallest(5, "model_risk")
surprises[["sub_grade", "model_risk", "loan_amnt", "annual_inc", "fico", "dti", "purpose", "emp_years"]].round(3)

## Save what the demo and later notebooks need

- the trained model
- each sub-grade's historical default rate (to translate the model's risk into "the grade this loan deserves")
- the 2015 test loans with the model's risk score

In [ ]:
Path("models").mkdir(exist_ok=True)
joblib.dump({"model": models[BEST], "features": FEATURES, "name": BEST}, "models/default_model.joblib")

subgrade_default = (
    train.groupby("sub_grade")["default"].mean().reindex(SUBGRADE_ORDER).dropna().rename("default_rate")
)
subgrade_default.to_csv("models/subgrade_default_rates.csv")

test.to_parquet("data/test_2015_scored.parquet", index=False)
results.to_csv("outputs/auc_results.csv")
money.to_csv("outputs/money_test.csv")
print("Saved model, sub-grade default rates, scored 2015 loans and result tables.")

In [ ]:
def deserved_subgrade(probability):
    # The sub-grade whose historical default rate is closest to this probability
    return (subgrade_default - probability).abs().idxmin()

example = test.iloc[0]
print(f"Example loan: graded {example['sub_grade']}, model risk {example['model_risk']:.1%}, "
      f"which looks like a {deserved_subgrade(example['model_risk'])} loan")